# 相関解析 & タイムコース表示
解析済みデータ（parsed-data）を使用し、乖離判定と色分け表示を行います。

In [17]:
%load_ext autoreload
%autoreload 2

import json
import sys
import traceback
from pathlib import Path
from datetime import datetime
import io
import re
import itertools
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import ipywidgets as widgets
from IPython.display import display, clear_output

from openpyxl import load_workbook, Workbook
from openpyxl.drawing.image import Image as XLImage

# プロジェクトルートとソースディレクトリのパス設定
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from common.csv_loader import load_parsed_for_analysis
from common.analysis_utils import (
    setup_japanese_font, pick_col, normalize_group_col, detect_value_cols,
    safe_name, make_output_dirs, save_table_csv, apply_value_range_filter,
    pearson_r, regression_fit_info, plot_suite, write_df_to_sheet,
    groupwise_stats, compute_pair_sample_metrics, make_metric_definitions_df, classify_outlier_level, insert_images_into_excel,
    REGRESSION_METHODS_ALL, REGRESSION_LABELS, SHEET_PLOTS, SHEET_SUMMARY, 
    SHEET_GROUP_STATS, SHEET_OUTLIERS, SHEET_SAMPLE_METRICS, SHEET_METRIC_DEFINITIONS,
    OUT_SUFFIX, ID_COL_CANDIDATES, GROUP_COL_CANDIDATES, VALUE_PREFIXES
)

OUTPUT_ROOT = PROJECT_ROOT / "data" / "export"
setup_japanese_font()

In [18]:
_state = {
    "df": None, 
    "id_col": None, 
    "group_col": None, 
    "value_cols": None, 
    "parsed_dir": None, 
    "metadata": None, 
    "profile_df": None, 
    "ref_outlier_map": None,
    "analysis_results": None
}

def discover_latest_parsed_dir(parsed_root=None):
    root = PROJECT_ROOT
    parsed_root = Path(parsed_root) if parsed_root is not None else root / "data" / "parsed-data"
    if not parsed_root.is_absolute():
        parsed_root = root / parsed_root
    
    if parsed_root.is_dir() and (parsed_root / "measurement.parquet").exists() and (parsed_root / "metadata.json").exists():
        return parsed_root
    
    if not parsed_root.exists():
        raise FileNotFoundError(f"Directory does not exist: {parsed_root}")
        
    candidates = [
        p for p in parsed_root.iterdir()
        if p.is_dir() and (p / "measurement.parquet").exists() and (p / "metadata.json").exists()
    ]
    if not candidates:
        raise FileNotFoundError(f"No valid parsed-data found under: {parsed_root}")
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

def load_parsed_data_for_notebook(parsed_dir=None):
    parsed_dir = discover_latest_parsed_dir(parsed_dir)
    measurement_df, profile_df, metadata, prescription_columns = load_parsed_for_analysis(parsed_dir)
    
    measurement_df = measurement_df.copy()
    if "SampleID" not in measurement_df.columns:
        if "SID" in measurement_df.columns:
            measurement_df["SampleID"] = measurement_df["SID"].astype(str)
        elif "依頼No." in measurement_df.columns:
            measurement_df["SampleID"] = measurement_df["依頼No."].astype(str)
        else:
            measurement_df["SampleID"] = measurement_df.index.astype(str)

    id_col = pick_col(measurement_df, ID_COL_CANDIDATES, default="SID")
    group_col_raw = pick_col(measurement_df, GROUP_COL_CANDIDATES, default=None)
    group_col = normalize_group_col(measurement_df, group_col_raw)

    value_cols = [c for c in prescription_columns if c in measurement_df.columns]
    if not value_cols:
        value_cols = detect_value_cols(measurement_df, id_col, group_col, prefixes=VALUE_PREFIXES)
    value_cols = [c for c in value_cols if c in measurement_df.columns and not str(c).endswith("_FLAG")]

    return measurement_df, profile_df, metadata, parsed_dir, id_col, group_col, value_cols

In [19]:
# ============================================================
# UI Widgets Definition
# ============================================================
help_html = widgets.HTML("<div style='border:1px solid #ccc; padding:12px; border-radius:8px; background:#fafafa; line-height:1.6;'>"
"<b>このツールの目的</b><br>"
"解析済みデータ（parsed-data）から相関・回帰・Bland–Altman・残差・乖離候補を確認し、Excelに出力します。<br>"
"また、検体ごとのタイムコース反応（吸光度変化）を確認できます。</div>")

input_dir_dd = widgets.Dropdown(options=[("(未選択)", "(未選択)")], value="(未選択)", description="解析対象")
refresh_dir_btn = widgets.Button(description="候補更新", button_style="info")

mode_dd = widgets.Dropdown(
    options=[("全組合せ", "all"), ("隣同士のみ", "adjacent"), ("基準処方 vs その他", "baseline")],
    value="baseline", description="モード")

baseline_sel = widgets.SelectMultiple(options=[], value=(), description="基準処方", rows=8)
all_reg_ck = widgets.Checkbox(value=False, description="全回帰法で出力")
reg_dd = widgets.Dropdown(
    options=[("OLS（最小二乗）", "OLS"), ("Deming（両軸誤差）", "Deming"), 
             ("Theil-Sen（ロバスト）", "TheilSen"), ("Passing-Bablok（ノンパラメトリック）", "PassingBablok")],
    value="PassingBablok", description="回帰法")

deming_lambda_val = widgets.FloatText(value=1.0, description="λ(Deming)")
best_mode_val = widgets.ToggleButtons(options=[("|r|最大", "abs"), ("r最大（正相関優先）", "pos")], value="pos", description="ベスト")
z_thresh_val = widgets.FloatSlider(value=3.5, min=1.5, max=8.0, step=0.1, description="乖離z(MAD)")
label_top_val = widgets.IntSlider(value=8, min=0, max=30, step=1, description="ラベル数")

use_range_ck = widgets.Checkbox(value=False, description="対象範囲絞り")
range_min_txt = widgets.FloatText(value=0.0, description="下限")
range_max_txt = widgets.FloatText(value=100.0, description="上限")

show_py_ck = widgets.Checkbox(value=True, description="Python上に表示")
max_show_val = widgets.IntSlider(value=6, min=0, max=30, step=1, description="表示上限")

ref_pair_x_dd = widgets.Dropdown(options=[("(未選択)", "(未選択)")], value="(未選択)", description="乖離基準X")
ref_pair_y_dd = widgets.Dropdown(options=[("(未選択)", "(未選択)")], value="(未選択)", description="乖離基準Y")

load_btn = widgets.Button(description="①データ読み込み", button_style="primary")
analyze_btn = widgets.Button(description="②解析実行", button_style="success")
export_btn = widgets.Button(description="③Excel出力", button_style="warning")

status = widgets.Output()
plots_out = widgets.Output()

# --- Time Course Specific Widgets ---
tc_item_dd = widgets.Dropdown(description='表示項目')
tc_conc_range_slider = widgets.FloatRangeSlider(
    value=[0, 1000], min=0, max=1000, step=0.1, description='濃度範囲', 
    layout=widgets.Layout(width='400px'), readout_format='.1f')
tc_outlier_dd = widgets.Dropdown(
    options=[("全て", "all"), ("乖離のみ", "outlier"), ("非乖離のみ", "normal")], 
    value="all", description="乖離選択")
tc_baseline_time_dd = widgets.Dropdown(description="基準時間(秒)", options=[("(未選択)", None)], value=None)

tc_show_btn = widgets.Button(description='タイムコース表示', button_style='info')
tc_out = widgets.Output()

In [20]:
def refresh_input_dir_list():
    try:
        parsed_root = PROJECT_ROOT / "data" / "parsed-data"
        if not parsed_root.exists():
            input_dir_dd.options = [("(フォルダ未作成)", "(未選択)")]
            return
        
        dirs = [p for p in sorted(parsed_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True) 
                if p.is_dir() and (p / "measurement.parquet").exists()]
        
        if dirs:
            input_dir_dd.options = [(p.name, str(p)) for p in dirs]
            input_dir_dd.value = str(dirs[0])
        else:
            input_dir_dd.options = [("(対象データなし)", "(未選択)")]
    except Exception:
        with status: 
            clear_output()
            print("ディレクトリ候補の更新に失敗しました。")
            print(traceback.format_exc())

def update_tc_items():
    if _state['profile_df'] is not None:
        items = list(_state['profile_df']['項目名'].unique())
        tc_item_dd.options = items
        if items: tc_item_dd.value = items[0]
        
        times = sorted(_state['profile_df']['時間'].unique())
        tc_baseline_time_dd.options = [("(生データ表示)", None)] + [(f"{t:.1f}s", t) for t in times]
        tc_baseline_time_dd.value = None

def on_tc_item_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        item = change['new']
        df = _state.get("df")
        if df is not None and item in df.columns:
            vals = pd.to_numeric(df[item], errors='coerce').dropna()
            if not vals.empty:
                vmin, vmax = float(vals.min()), float(vals.max())
                tc_conc_range_slider.min = vmin
                tc_conc_range_slider.max = vmax
                tc_conc_range_slider.value = [vmin, vmax]

tc_item_dd.observe(on_tc_item_change, names='value')

def load_action(_=None):
    with status:
        clear_output()
        target = None if input_dir_dd.value == "(未選択)" else input_dir_dd.value
        try:
            df, profile_df, metadata, parsed_dir, id_col, group_col, value_cols = load_parsed_data_for_notebook(target)
            _state.update({"df": df, "profile_df": profile_df, "metadata": metadata, "parsed_dir": parsed_dir, 
                           "id_col": id_col, "group_col": group_col, "value_cols": value_cols, "analysis_results": None})
            
            baseline_sel.options = value_cols
            if value_cols: baseline_sel.value = (value_cols[0],)
            
            ref_pair_x_dd.options = ["(未選択)"] + value_cols
            ref_pair_y_dd.options = ["(未選択)"] + value_cols
            if len(value_cols) >= 2:
                ref_pair_x_dd.value = value_cols[0]
                ref_pair_y_dd.value = value_cols[1]
            
            print(f"読み込み完了: {parsed_dir.name}")
            print(f"ID列: {id_col}, 比較列数: {len(value_cols)}")
            update_tc_items()
            on_tc_item_change({'type': 'change', 'name': 'value', 'new': tc_item_dd.value})
        except Exception:
            print(f"データ読み込みエラー:\n{traceback.format_exc()}")

def run_analysis_action(_=None):
    with status: clear_output()
    with plots_out: clear_output()
    
    try:
        if _state["df"] is None: 
            load_action()
            if _state["df"] is None: return
            
        df, id_col, group_col, value_cols, parsed_dir = _state["df"], _state["id_col"], _state["group_col"], _state["value_cols"], _state["parsed_dir"]
        
        if mode_dd.value == "adjacent": pairs = list(zip(value_cols[:-1], value_cols[1:]))
        elif mode_dd.value == "all": pairs = list(itertools.combinations(value_cols, 2))
        else:
            bases = list(baseline_sel.value) if baseline_sel.value else [value_cols[0]]
            pairs = [(b, c) for b in bases for c in value_cols if c != b]

        methods = REGRESSION_METHODS_ALL if all_reg_ck.value else [reg_dd.value]
        lam = deming_lambda_val.value
        z_thresh = z_thresh_val.value
        
        summary_rows, outlier_tables, sample_metric_tables, figures = [], [], [], []
        shown = 0
        run_metadata = _state["metadata"].copy() if _state["metadata"] else {}
        
        # --- Pre-calculate Reference Outliers for Coloring ---
        ref_outlier_map = {} 
        ref_x, ref_y = ref_pair_x_dd.value, ref_pair_y_dd.value
        if ref_x != "(未選択)" and ref_y != "(未選択)" and ref_x != ref_y:
            df_ref = apply_value_range_filter(df, ref_x, ref_y, use_range=use_range_ck.value, lo=range_min_txt.value, hi=range_max_txt.value)
            sub_ref = df_ref[[ref_x, ref_y]].dropna()
            if len(sub_ref) >= 2:
                xr, yr = sub_ref[ref_x].astype(float).values, sub_ref[ref_y].astype(float).values
                ar, br, _ = regression_fit_info(xr, yr, method=reg_dd.value, deming_lambda=lam)
                metrics_ref = compute_pair_sample_metrics(df_ref, id_col, group_col, ref_x, ref_y, ar, br, z_thresh=z_thresh)
                for _, row in metrics_ref.iterrows():
                    ref_outlier_map[str(row[id_col])] = row["outlier_level"]

        for method in methods:
            for xcol, ycol in pairs:
                df_pair = apply_value_range_filter(df, xcol, ycol, use_range=use_range_ck.value, lo=range_min_txt.value, hi=range_max_txt.value)
                sub = df_pair[[xcol, ycol]].dropna()
                if len(sub) < 2: continue
                
                x, y = sub[xcol].astype(float).values, sub[ycol].astype(float).values
                r = pearson_r(x, y)
                a, b, fit_info = regression_fit_info(x, y, method=method, deming_lambda=lam)
                
                pair_key = f"{xcol}_vs_{ycol}"
                metrics_df = compute_pair_sample_metrics(df_pair, id_col, group_col, xcol, ycol, a, b, z_thresh=z_thresh)
                
                color_list = []
                for _, row in metrics_df.iterrows():
                    sid = str(row[id_col])
                    sample_meta = run_metadata.setdefault(sid, {})
                    outliers_meta = sample_meta.setdefault("outliers", {})
                    
                    z_mad = row.get("z_MAD", np.nan)
                    level = classify_outlier_level(abs(z_mad), thresh=z_thresh) if np.isfinite(z_mad) else "none"
                    outliers_meta[pair_key] = {"level": level, "z_MAD": float(z_mad) if np.isfinite(z_mad) else None}
                    
                    # Determine color based on reference outlier map if available
                    target_level = ref_outlier_map.get(sid, "none") if ref_outlier_map else level
                    
                    if target_level == "strong_candidate": color_list.append("red")
                    elif target_level == "candidate": color_list.append("orange")
                    elif target_level == "mild_candidate": color_list.append("yellow")
                    else: color_list.append("#1f77b4")

                fig, used_sub, flagged, bias, loa = plot_suite(
                    df=df_pair, id_col=id_col, group_col=group_col, xcol=xcol, ycol=ycol,
                    method=method, lam=lam, a=a, b=b, r=r, fit_info=fit_info,
                    z_thresh=z_thresh, outlier_label_top=label_top_val.value,
                    fig_width=16, fig_height=10, dpi=100, external_colors=color_list
                )
                
                if fig:
                    if show_py_ck.value and shown < max_show_val.value:
                        with plots_out: display(fig)
                        shown += 1
                    figures.append((fig, method, xcol, ycol))

                summary_rows.append({"regression": method, "X": xcol, "Y": ycol, "n": len(x), "r": r, 
                                     "slope": a, "intercept": b, "BA_bias": bias, "n_outliers": len(flagged)})
                if not metrics_df.empty: sample_metric_tables.append(metrics_df.assign(regression=method, X=xcol, Y=ycol))
                if not flagged.empty: outlier_tables.append(flagged.assign(regression=method, X=xcol, Y=ycol))

        _state["analysis_results"] = {
            "run_metadata": run_metadata,
            "ref_outlier_map": ref_outlier_map,
            "summary_rows": summary_rows,
            "sample_metric_tables": sample_metric_tables,
            "outlier_tables": outlier_tables,
            "figures": figures
        }
        _state["metadata_enhanced"] = run_metadata
        _state["ref_outlier_map"] = ref_outlier_map
        
        with status: print(f"解析完了! {len(summary_rows)}件のペアを処理しました。内容を確認後、必要であれば『Excel出力』を押してください。")
    except Exception:
        with status: print(f"解析実行エラー:\n{traceback.format_exc()}")

def export_action(_=None):
    with status: clear_output()
    if _state["analysis_results"] is None:
        with status: print("先に『解析実行』を行ってください。"); return
    
    try:
        res = _state["analysis_results"]
        parsed_dir = _state["parsed_dir"]
        
        dirs = make_output_dirs(OUTPUT_ROOT, input_stem=parsed_dir.name)
        img_paths = []
        
        for fig, method, xcol, ycol in res["figures"]:
            png = dirs["plots"] / f"QC_{safe_name(method)}_{safe_name(ycol)}_vs_{safe_name(xcol)}.png"
            fig.savefig(png, bbox_inches="tight")
            img_paths.append(png)

        with open(parsed_dir / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(res["run_metadata"], f, indent=2, ensure_ascii=False)

        output_xlsx = dirs["excel"] / f"{parsed_dir.name}{OUT_SUFFIX}.xlsx"
        wb = Workbook(); wb.save(output_xlsx)
        
        if img_paths:
            insert_images_into_excel(input_xlsx=output_xlsx, output_xlsx=output_xlsx, image_paths=img_paths, plot_sheet=SHEET_PLOTS)
        
        if res["summary_rows"]: write_df_to_sheet(output_xlsx, pd.DataFrame(res["summary_rows"]), SHEET_SUMMARY)
        if res["sample_metric_tables"]: write_df_to_sheet(output_xlsx, pd.concat(res["sample_metric_tables"]), SHEET_SAMPLE_METRICS)
        if res["outlier_tables"]: write_df_to_sheet(output_xlsx, pd.concat(res["outlier_tables"]), SHEET_OUTLIERS)
        
        with status: print(f"Excel保存完了! 出力先: {output_xlsx}")
    except Exception:
        with status: print(f"出力エラー:\n{traceback.format_exc()}")

def plot_time_course_action(_=None):
    with tc_out:
        clear_output()
        try:
            df = _state.get("df")
            profile_df = _state.get("profile_df")
            metadata = _state.get("metadata_enhanced") or _state.get("metadata")
            ref_outlier_map = _state.get("ref_outlier_map")
            id_col = _state.get("id_col")
            
            if profile_df is None or df is None: 
                print("データが読み込まれていません。"); return
            
            item_name = tc_item_dd.value
            if not item_name: 
                print("表示項目が選択されていません。"); return
            
            # --- Filter by Concentration Range ---
            cmin, cmax = tc_conc_range_slider.value
            df_filtered = df[(pd.to_numeric(df[item_name], errors='coerce') >= cmin) & 
                             (pd.to_numeric(df[item_name], errors='coerce') <= cmax)]
            allowed_sids = set(df_filtered[id_col].astype(str).tolist())
            
            id_mapping = {}
            if "依頼No." in df.columns and id_col in df.columns:
                id_mapping = dict(zip(df["依頼No."].astype(str), df[id_col].astype(str)))
            
            # --- Filter by Outlier Status ---
            filter_mode = tc_outlier_dd.value # 'all', 'outlier', 'normal'
            
            df_item = profile_df[profile_df["項目名"] == item_name]
            fig, ax = plt.subplots(figsize=(12, 7))
            
            baseline_time = tc_baseline_time_dd.value
            
            plotted_count = 0
            for sid, gdf in df_item.groupby("依頼No."):
                sid_str = str(sid)
                mapped_id = id_mapping.get(sid_str, sid_str)
                if mapped_id not in allowed_sids: continue
                
                color, lw, alpha = "#1f77b4", 1.0, 0.3
                
                level = "none"
                if ref_outlier_map and mapped_id in ref_outlier_map:
                    level = ref_outlier_map[mapped_id]
                elif metadata and mapped_id in metadata and "outliers" in metadata[mapped_id]:
                    levels = [v["level"] for v in metadata[mapped_id]["outliers"].values()]
                    if "strong_candidate" in levels: level = "strong_candidate"
                    elif "candidate" in levels: level = "candidate"
                    elif "mild_candidate" in levels: level = "mild_candidate"
                
                is_outlier = level in ["strong_candidate", "candidate", "mild_candidate"]
                if filter_mode == "outlier" and not is_outlier: continue
                if filter_mode == "normal" and is_outlier: continue
                
                if level == "strong_candidate": color, lw, alpha = "red", 2.5, 0.9
                elif level == "candidate": color, lw, alpha = "orange", 2.0, 0.8
                elif level == "mild_candidate": color, lw, alpha = "yellow", 1.5, 0.7
                
                times = gdf["時間"].values
                abs_vals = gdf["吸光度"].values
                
                # --- Baseline Adjustment ---
                if baseline_time is not None:
                    idx = np.argmin(np.abs(times - baseline_time))
                    base_abs = abs_vals[idx]
                    abs_vals = abs_vals - base_abs
                
                ax.plot(times, abs_vals, color=color, linewidth=lw, alpha=alpha)
                plotted_count += 1

            ax.set_title(f"タイムコース反応: {item_name} (プロット数: {plotted_count}, 基準時間: {baseline_time if baseline_time is not None else 'None'}s)")
            ax.set_xlabel("時間(秒)"); ax.set_ylabel("吸光度"); ax.grid(True, alpha=0.3)
            if baseline_time is not None: ax.axvline(baseline_time, color='black', linestyle='--', alpha=0.5)
            plt.show()
        except Exception:
            print(f"タイムコース表示エラー:\n{traceback.format_exc()}")

In [21]:
# イベントの紐付け
refresh_dir_btn.on_click(lambda _: refresh_input_dir_list())
load_btn.on_click(load_action)
analyze_btn.on_click(run_analysis_action)
export_btn.on_click(export_action)
tc_show_btn.on_click(plot_time_course_action)

# 初期表示用処理を確実に実行
try:
    refresh_input_dir_list()
except Exception:
    pass

display(widgets.VBox([
    widgets.HTML("<b>①フォルダ選択 → ②読み込み → ③解析実行 → ④Excel出力(任意)</b>"),
    help_html,
    widgets.HBox([input_dir_dd, refresh_dir_btn]),
    widgets.HBox([mode_dd, all_reg_ck, reg_dd, deming_lambda_val]),
    widgets.HBox([baseline_sel, widgets.VBox([best_mode_val, z_thresh_val, label_top_val])]),
    widgets.HBox([use_range_ck, range_min_txt, range_max_txt]),
    widgets.HTML("<b>乖離判定の基準（このペアで赤い検体を他でも赤く表示）</b>"),
    widgets.HBox([ref_pair_x_dd, ref_pair_y_dd]),
    widgets.HBox([show_py_ck, max_show_val]),
    widgets.HBox([load_btn, analyze_btn, export_btn]),
    status, plots_out,
    widgets.HTML("<hr><b>タイムコース反応表示設定</b>"),
    widgets.HBox([tc_item_dd, tc_conc_range_slider]),
    widgets.HBox([tc_outlier_dd, tc_baseline_time_dd, tc_show_btn]),
    tc_out
]))
